# Transformer Self-Attention: From Scratch to TensorFlow

**Understanding attention mathematically, implementing it manually, and benchmarking it against a built-in implementation.**

This is a compact educational research project: it creates an original synthetic sequence-classification dataset, implements scaled dot-product self-attention with native TensorFlow operations, builds a matched Keras `MultiHeadAttention` version, trains both classifiers, and evaluates their numerical agreement, predictive quality, and runtime.

> **Scope note:** The dataset is synthetic by design. It tests whether the models can learn a small context-pair rule; it is not evidence of real-world NLP capability. The notebook measures results when it is run. It never hard-codes training scores or invents a winner.

### Learning objectives
- Understand token embeddings, positional information, queries, keys, values, scaling, softmax, and masks.
- Implement self-attention from first principles and verify the tensor math.
- Compare the implementation with TensorFlow/Keras `MultiHeadAttention` under compatible weights.
- Train two comparable classifiers on identical splits and evaluate them with accuracy, macro F1, precision, recall, and confusion matrices.
- Explore attention maps, masking, sequence length, embedding dimension, and inference timing.
- Export interactive Plotly charts, model artifacts, metrics, predictions, and reproducibility notes.

### Notebook map
Environment → synthetic data → visual inspection → mathematical walkthrough → manual layer → built-in layer → unit/parity tests → matched classifiers → training → evaluation → attention analysis → benchmarks and ablations → export.

## 1. Environment setup

The notebook uses Python, TensorFlow/Keras, NumPy, Pandas, scikit-learn, and Plotly. TensorFlow is a documented runtime dependency; if it is missing, install it into the active environment and restart the kernel before running the notebook. Static PNG export is optional. Interactive HTML export remains the primary, dependency-light output format.

In [1]:
from pathlib import Path
import json
import math
import os
import platform
import random
import time
import warnings

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for this notebook. Install a TensorFlow build "
        "compatible with your Python version, restart the kernel, and rerun."
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path.cwd()
IMAGE_DIR = PROJECT_DIR / "images"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
for directory in (IMAGE_DIR, OUTPUT_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Python:       {platform.python_version()}")
print(f"Platform:     {platform.platform()}")
print(f"TensorFlow:   {tf.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Plotly:       {plotly.__version__}")
print(f"Devices:      {tf.config.list_physical_devices()}")
print(f"Project dir:  {PROJECT_DIR.resolve()}")

Python:       3.13.15
Platform:     Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow:   2.21.0
NumPy:        2.1.3
Pandas:       2.2.3
Plotly:       5.24.1
Devices:      [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Project dir:  /content


### Reusable figure export helper

Every Plotly figure is displayed inline and written to `images/` as self-contained interactive HTML. The helper also attempts PNG export through Plotly's supported image-export mechanism. If `kaleido` or another required static-export dependency is unavailable, it records a warning and continues without failing the notebook.

In [2]:
PNG_EXPORT_AVAILABLE = True

def save_plotly_figure(fig, filename, *, show=True, export_png=True):
    """Save an interactive HTML chart and optionally attempt a static PNG export."""
    global PNG_EXPORT_AVAILABLE
    name = Path(filename).stem
    html_path = IMAGE_DIR / f"{name}.html"
    fig.write_html(html_path, include_plotlyjs=True, full_html=True)
    png_path = IMAGE_DIR / f"{name}.png"
    if export_png and PNG_EXPORT_AVAILABLE:
        try:
            fig.write_image(png_path, scale=2)
        except Exception as exc:
            PNG_EXPORT_AVAILABLE = False
            warnings.warn(
                f"PNG export skipped (first attempt): {type(exc).__name__}: {exc}. "
                "HTML was saved successfully. Install a compatible Kaleido version "
                "to enable static exports. Further PNG attempts are disabled for this run.", RuntimeWarning
            )
    if show:
        fig.show()
    return {"html": str(html_path.relative_to(PROJECT_DIR)),
            "png": str(png_path.relative_to(PROJECT_DIR)) if png_path.exists() else None}

PLOT_TEMPLATE = "plotly_white"
PLOT_FONT = "Arial, sans-serif"

## 2. Create an original synthetic dataset

### Task definition

The dataset contains five classes: **animals and pets**, **food and fruits**, **people and occupations**, **nature and space**, and **emotions and actions**. Each sequence contains two relational cue tokens plus context-independent distractors and neutral connective words. The target is determined by the *pair* of cues, not by a single cue:

| Class | Relational cue pair |
|---|---|
| Animals and pets | `pet` + `calm` |
| Food and fruits | `calm` + `harvest` |
| People and occupations | `harvest` + `shift` |
| Nature and space | `shift` + `orbit` |
| Emotions and actions | `orbit` + `pet` |

Each cue appears in two classes, so no individual cue uniquely determines the label. Topic words are sampled as distractors independently of the class. This creates a controlled context-pair task, not natural language. Sequence order is shuffled and lengths vary from 8 to 16 tokens.

In [3]:
CLASS_NAMES = [
    "Animals & Pets",
    "Food & Fruits",
    "People & Occupations",
    "Nature & Space",
    "Emotions & Actions",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i: name for name, i in CLASS_TO_ID.items()}

CUE_PAIRS = {
    0: ("pet", "calm"),
    1: ("calm", "harvest"),
    2: ("harvest", "shift"),
    3: ("shift", "orbit"),
    4: ("orbit", "pet"),
}

TOPIC_WORDS = [
    "king", "queen", "man", "woman", "dog", "cat", "animal", "companion",
    "apple", "fruit", "sweet", "food", "doctor", "nurse", "hospital", "health",
    "sun", "moon", "star", "space", "happy", "sad", "emotion", "joy",
    "run", "walk", "move", "action", "red", "blue", "green", "color",
    "teacher", "student", "school", "learn", "water", "river", "ocean", "liquid",
    "music", "book", "garden", "cloud", "family", "bright", "quiet", "gentle",
]
NEUTRAL_WORDS = ["the", "and", "with", "near", "inside", "while", "around", "beside", "often", "today"]
ALL_CUE_WORDS = sorted({word for pair in CUE_PAIRS.values() for word in pair})


def generate_synthetic_sequences(n_samples=12_000, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    # Equal class sampling makes macro metrics interpretable and reduces imbalance as a confound.
    labels = np.arange(n_samples) % len(CLASS_NAMES)
    rng.shuffle(labels)
    for sample_id, label in enumerate(labels):
        seq_len = int(rng.integers(8, 17))
        cues = list(CUE_PAIRS[int(label)])
        distractor_count = seq_len - len(cues)
        distractors = rng.choice(TOPIC_WORDS + NEUTRAL_WORDS, size=distractor_count, replace=True).tolist()
        tokens = cues + distractors
        rng.shuffle(tokens)
        rows.append({"sample_id": sample_id, "tokens": tokens, "label_id": int(label),
                     "label": CLASS_NAMES[int(label)], "sequence_length": len(tokens)})
    return pd.DataFrame(rows)

raw_df = generate_synthetic_sequences(n_samples=12_000, seed=SEED)
assert len(raw_df) == 12_000
assert set(raw_df["label_id"].unique()) == set(range(len(CLASS_NAMES)))
assert raw_df["tokens"].map(len).between(8, 16).all()
assert all(set(CUE_PAIRS[int(row.label_id)]).issubset(set(row.tokens)) for row in raw_df.itertuples())

print(f"Rows: {len(raw_df):,}")
print(f"Class count: {raw_df['label'].nunique()}")
print(f"Length range: {raw_df['sequence_length'].min()}–{raw_df['sequence_length'].max()} tokens")
display(raw_df[["sample_id", "tokens", "label", "sequence_length"]].head(10))

Rows: 12,000
Class count: 5
Length range: 8–16 tokens


,sample_id,tokens,label,sequence_length
0,0,"[king, pet, calm, fruit, teacher, garden, gard...",Animals & Pets,13
1,1,"[health, doctor, companion, orbit, red, star, ...",Emotions & Actions,15
2,2,"[cat, teacher, water, dog, often, liquid, clou...",Food & Fruits,16
3,3,"[space, run, nurse, companion, garden, with, s...",People & Occupations,12
4,4,"[companion, shift, often, fruit, orbit, sweet,...",Nature & Space,10
5,5,"[with, green, happy, color, happy, king, pet, ...",Animals & Pets,15
6,6,"[book, blue, orbit, shift, book, sweet, beside...",Nature & Space,8
7,7,"[emotion, happy, sun, around, and, harvest, te...",Food & Fruits,13
8,8,"[sweet, family, health, school, emotion, quiet...",Emotions & Actions,10
9,9,"[school, queen, star, teacher, woman, orbit, a...",Nature & Space,16


### Validate that one cue is not enough

This check is deliberately simple and auditable: for every cue token, inspect the labels in examples that contain it. Each cue should occur in two classes, rather than mapping to exactly one label. Distractor tokens are sampled independently of class, so their frequency should not carry intended label information.

In [4]:
cue_label_counts = []
for cue in ALL_CUE_WORDS:
    containing = raw_df[raw_df["tokens"].map(lambda seq: cue in seq)]
    counts = containing["label"].value_counts()
    cue_label_counts.append({
        "cue": cue,
        "labels_observed": int(counts.size),
        "dominant_label_share": float(counts.max() / counts.sum()),
        "observations": int(counts.sum()),
    })
cue_validation = pd.DataFrame(cue_label_counts).sort_values("cue")
display(cue_validation)
assert (cue_validation["labels_observed"] >= 2).all()
assert (cue_validation["dominant_label_share"] < 0.70).all(), "A cue is unexpectedly label-dominant."

,cue,labels_observed,dominant_label_share,observations
0,calm,2,0.5,4800
1,harvest,2,0.5,4800
2,orbit,2,0.5,4800
3,pet,2,0.5,4800
4,shift,2,0.5,4800


## 3. Split first, then fit vocabulary on training data

We split raw examples before constructing the vocabulary. This avoids using validation/test token observations to fit preprocessing. The task vocabulary is largely shared by design, but any unseen validation/test token safely maps to `<UNK>`. `<PAD>` uses ID `0`; `<UNK>` uses ID `1`. The fixed maximum length is 18, above the generator's maximum, so truncation is not expected for generated data.

In [5]:
train_df, temp_df = train_test_split(
    raw_df, test_size=0.30, random_state=SEED, stratify=raw_df["label_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label_id"]
)

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MAX_LEN = 18
train_tokens = {token for sequence in train_df["tokens"] for token in sequence}
VOCAB = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token in sorted(train_tokens - {PAD_TOKEN, UNK_TOKEN}):
    VOCAB[token] = len(VOCAB)
ID_TO_TOKEN = {idx: token for token, idx in VOCAB.items()}
PAD_ID = VOCAB[PAD_TOKEN]
UNK_ID = VOCAB[UNK_TOKEN]


def encode_sequences(sequences, vocab=VOCAB, max_len=MAX_LEN):
    encoded = np.full((len(sequences), max_len), PAD_ID, dtype=np.int32)
    for row_idx, tokens in enumerate(sequences):
        token_ids = [vocab.get(token, UNK_ID) for token in tokens[:max_len]]
        encoded[row_idx, :len(token_ids)] = token_ids
    return encoded

X_train = encode_sequences(train_df["tokens"].tolist())
y_train = train_df["label_id"].to_numpy(dtype=np.int32)
X_val = encode_sequences(val_df["tokens"].tolist())
y_val = val_df["label_id"].to_numpy(dtype=np.int32)
X_test = encode_sequences(test_df["tokens"].tolist())
y_test = test_df["label_id"].to_numpy(dtype=np.int32)

split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "examples": [len(train_df), len(val_df), len(test_df)],
    "share": [len(train_df)/len(raw_df), len(val_df)/len(raw_df), len(test_df)/len(raw_df)],
})
print(f"Vocabulary size (including special tokens): {len(VOCAB)}")
print(f"PAD_ID={PAD_ID}; UNK_ID={UNK_ID}; MAX_LEN={MAX_LEN}")
display(split_summary)
assert len(train_df) + len(val_df) + len(test_df) == len(raw_df)
assert X_train.shape == (len(train_df), MAX_LEN)
assert X_val.shape == (len(val_df), MAX_LEN)
assert X_test.shape == (len(test_df), MAX_LEN)
assert np.all(X_train[:, 0] >= 0)
assert np.isin(y_train, np.arange(len(CLASS_NAMES))).all()

Vocabulary size (including special tokens): 65
PAD_ID=0; UNK_ID=1; MAX_LEN=18


,split,examples,share
0,Train,8400,0.70
1,Validation,1800,0.15
2,Test,1800,0.15


## 4. Dataset inspection and Plotly visualizations

The charts below document class balance, sequence lengths, token frequency, split sizes, and example sequences. These are descriptive views of the synthetic input; they are not model evaluation results.

In [6]:
class_counts = raw_df["label"].value_counts().reindex(CLASS_NAMES).rename_axis("class").reset_index(name="count")
fig = px.bar(class_counts, x="class", y="count", color="class", text="count",
             title="Synthetic Dataset — Class Distribution", template=PLOT_TEMPLATE,
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Target class", yaxis_title="Examples", showlegend=False)
fig.update_traces(textposition="outside")
save_plotly_figure(fig, "dataset_class_distribution")

/tmp/ipykernel_4280/2800535721.py:15: RuntimeWarning:

PNG export skipped (first attempt): ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
. HTML was saved successfully. Install a compatible Kaleido version to enable static exports. Further PNG attempts are disabled for this run.



{'html': 'images/dataset_class_distribution.html', 'png': None}

In [7]:
fig = px.histogram(raw_df, x="sequence_length", color="label", barmode="overlay", opacity=0.78,
                   nbins=9, title="Sequence Length Distribution", template=PLOT_TEMPLATE,
                   color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Tokens per sequence", yaxis_title="Number of examples")
save_plotly_figure(fig, "dataset_sequence_lengths")

{'html': 'images/dataset_sequence_lengths.html', 'png': None}

In [8]:
from collections import Counter
freq = Counter(token for seq in raw_df["tokens"] for token in seq)
token_freq = pd.DataFrame(freq.most_common(30), columns=["token", "frequency"])
fig = px.bar(token_freq.sort_values("frequency"), x="frequency", y="token", orientation="h",
             color="frequency", color_continuous_scale="Turbo",
             title="Most Frequent Tokens (Top 30)", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Token occurrences", yaxis_title="Token")
save_plotly_figure(fig, "dataset_token_frequency")

{'html': 'images/dataset_token_frequency.html', 'png': None}

In [9]:
fig = px.bar(split_summary, x="split", y="examples", color="split", text="examples",
             title="Train / Validation / Test Split Sizes", template=PLOT_TEMPLATE,
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Split", yaxis_title="Number of examples", showlegend=False)
fig.update_traces(textposition="outside")
save_plotly_figure(fig, "dataset_split_sizes")

{'html': 'images/dataset_split_sizes.html', 'png': None}

In [10]:
sample_rows = raw_df.sample(8, random_state=SEED).reset_index(drop=True)
max_show_tokens = max(sample_rows["sequence_length"])
z, hover_tokens = [], []
for tokens in sample_rows["tokens"]:
    row_ids = [VOCAB.get(t, UNK_ID) for t in tokens]
    z.append(row_ids + [np.nan] * (max_show_tokens - len(row_ids)))
    hover_tokens.append(tokens + [""] * (max_show_tokens - len(tokens)))
fig = go.Figure(go.Heatmap(
    z=z, x=[f"Position {i+1}" for i in range(max_show_tokens)],
    y=[f"{r.label} · sample {r.sample_id}" for r in sample_rows.itertuples()],
    text=hover_tokens, hovertemplate="%{y}<br>%{x}<br>Token: %{text}<extra></extra>",
    colorscale="Turbo", colorbar_title="Token ID", connectgaps=False
))
fig.update_layout(title="Example Sequences — Hover to Inspect Tokens", template=PLOT_TEMPLATE,
                  font_family=PLOT_FONT, xaxis_title="Token position", yaxis_title="Sequence")
save_plotly_figure(fig, "dataset_example_sequences")

{'html': 'images/dataset_example_sequences.html', 'png': None}

### What the dataset can and cannot show

- The class count is balanced on purpose; this reduces one source of noise in the comparison.
- The cue pair is the defining signal, while unrelated lexical distractors and random positions make the task less like a one-token lookup.
- Because the generator and target rule are artificial, high test accuracy would only show that the models learned this synthetic rule.
- Self-attention weights are useful for inspecting computation, but they are not definitive explanations of a model's reasoning.

## 5. Mathematical foundations

### Why attention?

A token embedding is initially a vector associated with a token. A contextual representation should also depend on other tokens in the same sequence. Self-attention lets each token construct a weighted combination of information from the sequence. For example, the representation of `pet` may be influenced by `calm`, `orbit`, or surrounding distractors depending on the input and learned projections.

### Queries, keys, and values

For input representations \(X\), learned linear projections produce:

\[
Q=XW_Q+b_Q,\qquad K=XW_K+b_K,\qquad V=XW_V+b_V.
\]

- **Query (Q):** what each position is looking for.
- **Key (K):** what each position offers for matching.
- **Value (V):** the information contributed if that position is attended to.

The score for query position \(i\) and key position \(j\) is a dot product \(q_i\cdot k_j\). Scores are scaled by \(\sqrt{d_k}\), then normalized with softmax over key positions. Finally, the normalized probabilities weight the value vectors.

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

Here \(M\) is an optional additive mask: valid key positions add zero, while masked keys receive a very negative score so their probabilities become effectively zero. The core unmasked form is:

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### Why divide by \(\sqrt{d_k}\)?

When query/key dimensions grow, dot products can grow in magnitude. Large logits can push softmax toward saturated probabilities and weaker gradients. Scaling helps keep score magnitudes in a more stable range.

### Shape guide

For batch size \(B\), sequence length \(T\), query/key dimension \(d_k\), and value dimension \(d_v\):

| Tensor | Shape |
|---|---|
| Input embeddings | `(B, T, d_model)` |
| Queries | `(B, T, d_k)` |
| Keys | `(B, T, d_k)` |
| Values | `(B, T, d_v)` |
| Scores | `(B, T, T)` |
| Attention probabilities | `(B, T, T)` |
| Contextual output | `(B, T, d_v)` |

### Positions, masks, and attention variants

- **Positional information** helps distinguish the same token at different positions; attention by itself is permutation-equivariant without positional signals.
- **Padding masks** prevent real query positions from assigning attention mass to padded key positions. Pooling must also ignore padded query positions.
- **Self-attention** uses Q, K, and V from the same sequence. **Cross-attention** uses queries from one sequence and keys/values from another.
- **Single-head attention** learns one set of projections. **Multi-head attention** runs several learned projection subspaces and combines their outputs.
- **Self-attention is not inherently causal.** A causal mask is needed when future tokens must not be visible, such as in autoregressive language modeling. This classification task uses non-causal encoder-style attention.

## 6. A hand-checkable numerical example

Use tiny matrices so every operation can be inspected. In this example, \(Q\) and \(K\) have dimension 2 and \(V\) has dimension 2. NumPy is used only for the worked calculation; the trainable model layers below use TensorFlow operations for forward passes and gradients.

In [11]:
Q_demo = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=np.float32)
K_demo = np.array([[1.0, 0.0], [1.0, 1.0]], dtype=np.float32)
V_demo = np.array([[10.0, 0.0], [0.0, 20.0]], dtype=np.float32)
raw_scores_demo = Q_demo @ K_demo.T
scaled_scores_demo = raw_scores_demo / np.sqrt(K_demo.shape[-1])
shifted = scaled_scores_demo - scaled_scores_demo.max(axis=-1, keepdims=True)
exp_scores = np.exp(shifted)
attention_demo = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
context_demo = attention_demo @ V_demo

print("Q =\n", Q_demo)
print("K =\n", K_demo)
print("V =\n", V_demo)
print("QK^T =\n", raw_scores_demo)
print("Scaled scores =\n", scaled_scores_demo)
print("Softmax probabilities =\n", attention_demo)
print("Context output =\n", context_demo)
assert np.allclose(attention_demo.sum(axis=-1), 1.0, atol=1e-6)

Q =
 [[1. 0.]
 [0. 1.]]
K =
 [[1. 0.]
 [1. 1.]]
V =
 [[10.  0.]
 [ 0. 20.]]
QK^T =
 [[1. 1.]
 [0. 1.]]
Scaled scores =
 [[0.70710678 0.70710678]
 [0.         0.70710678]]
Softmax probabilities =
 [[0.5        0.5       ]
 [0.33023845 0.66976155]]
Context output =
 [[ 5.         10.        ]
 [ 3.30238451 13.39523099]]


In [12]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Raw scores QKᵀ", "Scaled scores", "Attention probabilities"))
for col, matrix, colorscale in [(1, raw_scores_demo, "Viridis"), (2, scaled_scores_demo, "Cividis"), (3, attention_demo, "Turbo")]:
    fig.add_trace(go.Heatmap(z=matrix, text=np.round(matrix, 3), texttemplate="%{text}",
                             colorscale=colorscale, showscale=False, hovertemplate="row=%{y}<br>col=%{x}<br>value=%{z:.4f}<extra></extra>"), row=1, col=col)
fig.update_layout(title="Scaled Dot-Product Attention — Numerical Walkthrough", template=PLOT_TEMPLATE,
                  font_family=PLOT_FONT, height=430)
save_plotly_figure(fig, "attention_numerical_walkthrough")

{'html': 'images/attention_numerical_walkthrough.html', 'png': None}

## 7. Implement scaled dot-product attention from scratch

The next layer takes already projected Q, K, and V tensors. `key_mask` uses **True/1 = valid key** and **False/0 = masked key**. It supports `(batch, key_length)` masks and broadcastable `(batch, query_length, key_length)` masks. Attention probabilities are normalized across the final axis—the key positions.

In [13]:
@tf.keras.utils.register_keras_serializable(package="SelfAttentionTutorial")
class ScaledDotProductAttention(tf.keras.layers.Layer):
    """Scaled dot-product attention implemented with native TensorFlow operations."""
    def call(self, query, key, value, key_mask=None):
        query = tf.convert_to_tensor(query)
        key = tf.convert_to_tensor(key)
        value = tf.convert_to_tensor(value)
        tf.debugging.assert_equal(tf.shape(query)[-1], tf.shape(key)[-1],
                                  message="Query and key dimensions must match.")
        tf.debugging.assert_equal(tf.shape(key)[-2], tf.shape(value)[-2],
                                  message="Key and value sequence dimensions must match.")
        depth = tf.cast(tf.shape(key)[-1], query.dtype)
        scores = tf.matmul(query, key, transpose_b=True) / tf.sqrt(depth)
        if key_mask is not None:
            valid = tf.cast(key_mask, tf.bool)
            if valid.shape.rank == 2:
                valid = valid[:, tf.newaxis, :]
            scores = tf.where(valid, scores, tf.cast(-1e9, scores.dtype))
        weights = tf.nn.softmax(scores, axis=-1)
        context = tf.matmul(weights, value)
        return context, weights

attention_math = ScaledDotProductAttention(name="scaled_dot_product_attention")
context_tf, weights_tf = attention_math(
    tf.constant(Q_demo[None, ...]), tf.constant(K_demo[None, ...]), tf.constant(V_demo[None, ...])
)
print("Context shape:", context_tf.shape)
print("Weights shape:", weights_tf.shape)
print("Maximum NumPy/TensorFlow difference:", float(np.max(np.abs(context_tf.numpy()[0] - context_demo))))
assert np.allclose(context_tf.numpy()[0], context_demo, atol=1e-6)

Context shape: (1, 2, 2)
Weights shape: (1, 2, 2)
Maximum NumPy/TensorFlow difference: 2.6041510459151596e-07


### Test padding-mask behavior and gradients

Padding positions must receive negligible probability as keys. The following test also verifies finite outputs and finite gradients. In real data, every sequence here contains at least two non-padding tokens, so softmax never receives a row with *all* keys masked.

In [14]:
q_test = tf.Variable(tf.random.normal((2, 5, 8), seed=SEED))
k_test = tf.Variable(tf.random.normal((2, 5, 8), seed=SEED + 1))
v_test = tf.Variable(tf.random.normal((2, 5, 6), seed=SEED + 2))
valid_test = tf.constant([[True, True, True, False, False], [True, True, False, False, False]])
with tf.GradientTape() as tape:
    out_test, weights_test = attention_math(q_test, k_test, v_test, key_mask=valid_test)
    objective = tf.reduce_sum(tf.square(out_test))
grads = tape.gradient(objective, [q_test, k_test, v_test])
masked_probs = tf.where(valid_test[:, None, :], tf.zeros_like(weights_test), weights_test)
print("Output shape:", out_test.shape)
print("Attention shape:", weights_test.shape)
print("Max masked-key probability:", float(tf.reduce_max(masked_probs)))
print("All outputs finite:", bool(tf.reduce_all(tf.math.is_finite(out_test))))
print("All gradients finite:", all(g is not None and bool(tf.reduce_all(tf.math.is_finite(g))) for g in grads))
assert float(tf.reduce_max(masked_probs)) < 1e-7
assert bool(tf.reduce_all(tf.math.is_finite(out_test)))
assert all(g is not None and bool(tf.reduce_all(tf.math.is_finite(g))) for g in grads)

Output shape: (2, 5, 6)
Attention shape: (2, 5, 5)
Max masked-key probability: 0.0
All outputs finite: True
All gradients finite: True


## 8. Learnable Q/K/V projections: custom self-attention

Scaled dot-product attention alone does not learn Q/K/V projections; it only combines tensors that it receives. The `SelfAttention` layer below learns three dense transformations and an output projection. The output projection makes its parameterization comparable to `MultiHeadAttention`, which also projects the concatenated head output back into the target feature dimension.

In [15]:
@tf.keras.utils.register_keras_serializable(package="SelfAttentionTutorial")
class SelfAttention(tf.keras.layers.Layer):
    """Single-head learned self-attention with trainable Q/K/V and output projections."""
    def __init__(self, d_model, key_dim=None, value_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.key_dim = int(key_dim or d_model)
        self.value_dim = int(value_dim or self.key_dim)
        self.query_projection = tf.keras.layers.Dense(self.key_dim, use_bias=True, name="query_projection")
        self.key_projection = tf.keras.layers.Dense(self.key_dim, use_bias=True, name="key_projection")
        self.value_projection = tf.keras.layers.Dense(self.value_dim, use_bias=True, name="value_projection")
        self.output_projection = tf.keras.layers.Dense(self.d_model, use_bias=True, name="output_projection")
        self.scaled_attention = ScaledDotProductAttention(name="scaled_dot_product")

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "key_dim": self.key_dim, "value_dim": self.value_dim})
        return config

    def call(self, inputs, valid_tokens=None, return_attention=False, training=None):
        q = self.query_projection(inputs)
        k = self.key_projection(inputs)
        v = self.value_projection(inputs)
        context, weights = self.scaled_attention(q, k, v, key_mask=valid_tokens)
        output = self.output_projection(context)
        if return_attention:
            return output, weights
        return output

custom_demo_layer = SelfAttention(d_model=16, key_dim=16, value_dim=16, name="custom_self_attention_demo")
_demo_x = tf.random.normal((2, 7, 16), seed=SEED)
_demo_mask = tf.constant([[True, True, True, True, False, False, False], [True, True, True, True, True, False, False]])
_demo_out, _demo_weights = custom_demo_layer(_demo_x, valid_tokens=_demo_mask, return_attention=True)
print("Input shape:", _demo_x.shape)
print("Output shape:", _demo_out.shape)
print("Attention weights shape:", _demo_weights.shape)
print("Trainable parameters:", custom_demo_layer.count_params())

Input shape: (2, 7, 16)
Output shape: (2, 7, 16)
Attention weights shape: (2, 7, 7)
Trainable parameters: 1088


## 9. Built-in TensorFlow/Keras attention

The built-in version uses `tf.keras.layers.MultiHeadAttention(num_heads=1)`. It returns both the transformed output and attention scores when requested. The model uses one head for the primary benchmark so the experiment compares implementation choices rather than single-head versus multi-head capacity.

Keras' built-in layer can include query/key/value projections, biases, an output projection, and version-specific implementation details. We explicitly align these components for a numerical parity test instead of assuming that two superficially similar layers are equivalent.

In [16]:
@tf.keras.utils.register_keras_serializable(package="SelfAttentionTutorial")
class BuiltinSelfAttention(tf.keras.layers.Layer):
    """Single-head wrapper around Keras MultiHeadAttention with matching mask semantics."""
    def __init__(self, d_model, key_dim=None, value_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.key_dim = int(key_dim or d_model)
        self.value_dim = int(value_dim or self.key_dim)
        self.mha = tf.keras.layers.MultiHeadAttention(
            num_heads=1,
            key_dim=self.key_dim,
            value_dim=self.value_dim,
            output_shape=self.d_model,
            dropout=0.0,
            use_bias=True,
            name="keras_multi_head_attention",
        )

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "key_dim": self.key_dim, "value_dim": self.value_dim})
        return config

    def call(self, inputs, valid_tokens=None, return_attention=False, training=None):
        attention_mask = None
        if valid_tokens is not None:
            valid_tokens = tf.cast(valid_tokens, tf.bool)
            batch_size = tf.shape(inputs)[0]
            sequence_length = tf.shape(inputs)[1]
            attention_mask = tf.broadcast_to(
                valid_tokens[:, tf.newaxis, :],
                tf.stack([batch_size, sequence_length, sequence_length]),
            )
        output, weights = self.mha(
            query=inputs, value=inputs, key=inputs,
            attention_mask=attention_mask,
            return_attention_scores=True,
            training=training,
        )
        if return_attention:
            return output, weights
        return output

builtin_demo_layer = BuiltinSelfAttention(d_model=16, key_dim=16, value_dim=16, name="builtin_self_attention_demo")
_demo_builtin_out, _demo_builtin_weights = builtin_demo_layer(_demo_x, valid_tokens=_demo_mask, return_attention=True)
print("Built-in output shape:", _demo_builtin_out.shape)
print("Built-in attention scores shape (batch, heads, query, key):", _demo_builtin_weights.shape)
print("Built-in trainable parameters:", builtin_demo_layer.count_params())

Built-in output shape: (2, 7, 16)
Built-in attention scores shape (batch, heads, query, key): (2, 1, 7, 7)
Built-in trainable parameters: 1088


## 10. Unit tests and numerical parity

The first test checks the manual attention function against a direct TensorFlow reference. The second test aligns the custom layer's Q/K/V and output-projection parameters with the built-in layer. It uses Keras' current internal projection-layer attributes when available; these are not public API, so a version change may require a small adaptation. The notebook reports the measured error and only calls the layers numerically aligned if that error meets the stated tolerance.

In [17]:
# Direct reference calculation for manual scaled dot-product attention.
ref_q = tf.constant(np.array([[[1., 0.], [0., 1.]]], dtype=np.float32))
ref_k = tf.constant(np.array([[[1., 0.], [1., 1.]]], dtype=np.float32))
ref_v = tf.constant(np.array([[[10., 0.], [0., 20.]]], dtype=np.float32))
ref_scores = tf.matmul(ref_q, ref_k, transpose_b=True) / tf.sqrt(tf.cast(tf.shape(ref_k)[-1], tf.float32))
ref_weights = tf.nn.softmax(ref_scores, axis=-1)
ref_output = tf.matmul(ref_weights, ref_v)
manual_output, manual_weights = attention_math(ref_q, ref_k, ref_v)
manual_error = float(tf.reduce_max(tf.abs(ref_output - manual_output)))
print(f"PASS — manual attention vs TensorFlow reference; max abs error={manual_error:.8g}")
assert manual_error < 1e-6

PASS — manual attention vs TensorFlow reference; max abs error=0


In [18]:
def copy_custom_projection_weights_to_builtin(custom_layer, builtin_layer):
    """Copy compatible single-head projection weights into a built Keras MHA layer."""
    # Both layers must have been called once to create their weights.
    d_model = custom_layer.d_model
    key_dim = custom_layer.key_dim
    value_dim = custom_layer.value_dim
    custom_projections = [
        custom_layer.query_projection.get_weights(),
        custom_layer.key_projection.get_weights(),
        custom_layer.value_projection.get_weights(),
    ]
    target_layers = [builtin_layer.mha._query_dense, builtin_layer.mha._key_dense, builtin_layer.mha._value_dense]
    for (kernel, bias), target in zip(custom_projections, target_layers):
        target.set_weights([kernel.reshape(d_model, 1, -1), bias.reshape(1, -1)])
    out_kernel, out_bias = custom_layer.output_projection.get_weights()
    builtin_layer.mha._output_dense.set_weights([out_kernel.reshape(1, value_dim, d_model), out_bias])

parity_custom = SelfAttention(d_model=16, key_dim=16, value_dim=16, name="parity_custom")
parity_builtin = BuiltinSelfAttention(d_model=16, key_dim=16, value_dim=16, name="parity_builtin")
parity_x = tf.random.normal((3, 6, 16), seed=SEED + 10)
parity_mask = tf.constant([[True, True, True, True, False, False],
                           [True, True, True, False, False, False],
                           [True, True, True, True, True, False]])
_ = parity_custom(parity_x, valid_tokens=parity_mask, return_attention=True)
_ = parity_builtin(parity_x, valid_tokens=parity_mask, return_attention=True)
parity_status = "not attempted"
try:
    copy_custom_projection_weights_to_builtin(parity_custom, parity_builtin)
    out_custom, att_custom = parity_custom(parity_x, valid_tokens=parity_mask, return_attention=True)
    out_builtin, att_builtin = parity_builtin(parity_x, valid_tokens=parity_mask, return_attention=True)
    att_builtin = att_builtin[:, 0, :, :]
    output_diff = np.abs(out_custom.numpy() - out_builtin.numpy())
    attention_diff = np.abs(att_custom.numpy() - att_builtin.numpy())
    max_output_error = float(output_diff.max())
    mean_output_error = float(output_diff.mean())
    max_attention_error = float(attention_diff.max())
    mean_attention_error = float(attention_diff.mean())
    parity_tolerance = 2e-5
    parity_status = "PASS" if max_output_error < parity_tolerance and max_attention_error < parity_tolerance else "CHECK"
    print(f"{parity_status} — output max/mean abs error: {max_output_error:.8g} / {mean_output_error:.8g}")
    print(f"Attention max/mean abs error: {max_attention_error:.8g} / {mean_attention_error:.8g}")
    print(f"Tolerance: {parity_tolerance}")
except (AttributeError, ValueError, TypeError) as exc:
    max_output_error = mean_output_error = max_attention_error = mean_attention_error = None
    parity_status = f"SKIPPED — installed Keras projection layout differs: {type(exc).__name__}: {exc}"
    print(parity_status)

PASS — output max/mean abs error: 5.9604645e-07 / 4.1851308e-08
Attention max/mean abs error: 5.9604645e-08 / 6.8124524e-09
Tolerance: 2e-05


### Visualize numerical agreement

These plots compare actual tensors produced by the parity test. They are useful diagnostics, not a promise that every future Keras release will expose identical internals.

In [19]:
if "out_custom" in globals() and "out_builtin" in globals():
    flat_custom = out_custom.numpy().ravel()
    flat_builtin = out_builtin.numpy().ravel()
    max_n = min(4000, len(flat_custom))
    fig = px.scatter(x=flat_custom[:max_n], y=flat_builtin[:max_n], opacity=0.55,
                     title="Custom vs Built-In Output Values", labels={"x": "Custom output", "y": "Built-in output"},
                     template=PLOT_TEMPLATE)
    low = float(min(flat_custom[:max_n].min(), flat_builtin[:max_n].min()))
    high = float(max(flat_custom[:max_n].max(), flat_builtin[:max_n].max()))
    fig.add_trace(go.Scatter(x=[low, high], y=[low, high], mode="lines", name="Ideal y = x", line={"dash": "dash"}))
    save_plotly_figure(fig, "attention_output_agreement")

    fig = px.histogram(x=np.abs(flat_custom[:max_n] - flat_builtin[:max_n]), nbins=50,
                       title="Absolute Error Distribution — Matched Attention Outputs",
                       labels={"x": "Absolute error", "count": "Elements"}, template=PLOT_TEMPLATE)
    save_plotly_figure(fig, "attention_output_error_distribution")

    fig = go.Figure(go.Heatmap(z=np.abs(att_custom.numpy()[0] - att_builtin.numpy()[0]),
                               colorscale="Magma", colorbar_title="|Δ weight|"))
    fig.update_layout(title="Absolute Attention-Weight Difference (Parity Test)", template=PLOT_TEMPLATE,
                      xaxis_title="Key position", yaxis_title="Query position")
    save_plotly_figure(fig, "attention_weight_difference")

## 11. Inspect batch tensors and padding masks

`X_train` holds integer IDs with shape `(examples, MAX_LEN)`. The boolean valid-token mask is `token_id != PAD_ID`; it is passed to attention as a key mask and separately used by the pooling layer to exclude padding queries from the sequence representation.

In [20]:
BATCH_SIZE = 128
EMBED_DIM = 48
FF_DIM = 96
NUM_CLASSES = len(CLASS_NAMES)
LEARNING_RATE = 1e-3
MAX_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 3

example_batch = tf.convert_to_tensor(X_train[:4], dtype=tf.int32)
example_valid_mask = tf.not_equal(example_batch, PAD_ID)
example_token_embeddings = tf.keras.layers.Embedding(len(VOCAB), EMBED_DIM)(example_batch)
print("Token IDs:", example_batch.shape)
print("Valid-token mask:", example_valid_mask.shape)
print("Token embeddings:", example_token_embeddings.shape)
print("First sequence tokens:", [ID_TO_TOKEN.get(int(i), UNK_TOKEN) for i in example_batch[0].numpy() if int(i) != PAD_ID])

Token IDs: (4, 18)
Valid-token mask: (4, 18)
Token embeddings: (4, 18, 48)
First sequence tokens: ['orbit', 'river', 'fruit', 'quiet', 'around', 'gentle', 'action', 'learn', 'companion', 'bright', 'pet', 'near']


## 12. Two matched sequence classifiers

Both models use the same token embedding, trainable positional embedding, residual connections, Layer Normalization, feed-forward network, masked mean pooling, and five-class softmax head. The only intended difference is the attention implementation. The attention dimension equals `EMBED_DIM`; this makes the custom and built-in projection weights directly comparable for one head.

In [21]:
@tf.keras.utils.register_keras_serializable(package="SelfAttentionTutorial")
class AttentionClassifier(tf.keras.Model):
    def __init__(self, vocab_size, max_len, num_classes, embed_dim, ff_dim, attention_kind="custom", **kwargs):
        super().__init__(**kwargs)
        if attention_kind not in {"custom", "builtin"}:
            raise ValueError("attention_kind must be 'custom' or 'builtin'.")
        self.attention_kind = attention_kind
        self.vocab_size = int(vocab_size)
        self.max_len = int(max_len)
        self.num_classes = int(num_classes)
        self.embed_dim = int(embed_dim)
        self.ff_dim = int(ff_dim)
        self.token_embedding = tf.keras.layers.Embedding(vocab_size, embed_dim, name="token_embedding")
        self.position_embedding = tf.keras.layers.Embedding(max_len, embed_dim, name="position_embedding")
        if attention_kind == "custom":
            self.attention = SelfAttention(embed_dim, key_dim=embed_dim, value_dim=embed_dim, name="custom_attention")
        else:
            self.attention = BuiltinSelfAttention(embed_dim, key_dim=embed_dim, value_dim=embed_dim, name="builtin_attention")
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="attention_layer_norm")
        self.ffn1 = tf.keras.layers.Dense(ff_dim, activation="relu", name="ffn_expand")
        self.ffn2 = tf.keras.layers.Dense(embed_dim, name="ffn_project")
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="ffn_layer_norm")
        self.classifier = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")

    def get_config(self):
        config = super().get_config()
        config.update({"vocab_size": self.vocab_size, "max_len": self.max_len,
                       "num_classes": self.num_classes, "embed_dim": self.embed_dim,
                       "ff_dim": self.ff_dim, "attention_kind": self.attention_kind})
        return config

    def call(self, token_ids, training=False, return_attention=False):
        token_ids = tf.cast(token_ids, tf.int32)
        valid_tokens = tf.not_equal(token_ids, PAD_ID)
        positions = tf.range(tf.shape(token_ids)[1])[tf.newaxis, :]
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        if return_attention:
            attended, attention_scores = self.attention(
                x, valid_tokens=valid_tokens, return_attention=True, training=training
            )
        else:
            attended = self.attention(x, valid_tokens=valid_tokens, training=training)
            attention_scores = None
        x = self.norm1(x + attended)
        ff = self.ffn2(self.ffn1(x))
        x = self.norm2(x + ff)
        weights = tf.cast(valid_tokens[:, :, tf.newaxis], x.dtype)
        pooled = tf.reduce_sum(x * weights, axis=1) / tf.maximum(tf.reduce_sum(weights, axis=1), 1.0)
        probabilities = self.classifier(pooled)
        if return_attention:
            return probabilities, attention_scores
        return probabilities


def build_classifier(attention_kind, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = AttentionClassifier(
        vocab_size=len(VOCAB), max_len=MAX_LEN, num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM, ff_dim=FF_DIM, attention_kind=attention_kind,
        name=f"{attention_kind}_self_attention_classifier",
    )
    _ = model(tf.zeros((1, MAX_LEN), dtype=tf.int32), training=False)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model

custom_model = build_classifier("custom", seed=SEED)
builtin_model = build_classifier("builtin", seed=SEED)
print("Custom parameter count:", custom_model.count_params())
print("Built-in parameter count:", builtin_model.count_params())
print("Parameter difference:", builtin_model.count_params() - custom_model.count_params())

Custom parameter count: 23189
Built-in parameter count: 23189
Parameter difference: 0


### Architecture check

Parameter counts are measured after the layers have been built. If counts differ, that difference is reported instead of being ignored. Equal counts do not guarantee identical training trajectories because initialization, low-level kernels, and floating-point operation ordering can still differ.

In [22]:
summary_lines = []
for model in (custom_model, builtin_model):
    lines = []
    model.summary(print_fn=lines.append)
    summary_lines.append(f"MODEL: {model.name}\n" + "\n".join(lines))
    print("\n".join(lines))
    print()
assert custom_model.output_shape if hasattr(custom_model, "output_shape") else True

Model: "custom_self_attention_classifier"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_embedding (Embedding)     │ (1, 18, 48)            │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ position_embedding (Embedding)  │ (1, 18, 48)            │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_attention                │ ?                      │         9,408 │
│ (SelfAttention)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_norm            │ (1, 18, 48)            │            96 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────

Model: "builtin_self_attention_classifier"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_embedding (Embedding)     │ (1, 18, 48)            │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ position_embedding (Embedding)  │ (1, 18, 48)            │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ builtin_attention               │ ?                      │         9,408 │
│ (BuiltinSelfAttention)          │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_norm            │ (1, 18, 48)            │            96 │
│ (LayerNormalization)            │                        │               │
├────────────────────────────────

## 13. Train both models independently

Training uses the same splits, optimizer family, learning rate, loss, batch size, maximum epochs, and callback policy. Early stopping restores the best validation-loss weights. The runtime includes model training and validation but excludes dataset generation. Run this section in a TensorFlow environment with sufficient memory; no downloaded dataset is required.

In [23]:
def train_model(model, run_name):
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1,
        ),
    ]
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=2,
    )
    duration = time.perf_counter() - start
    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_frame.to_csv(OUTPUT_DIR / f"{run_name}_training_history.csv", index=False)
    return history, duration

custom_history, custom_training_seconds = train_model(custom_model, "custom_attention")
builtin_history, builtin_training_seconds = train_model(builtin_model, "builtin_attention")
print(f"Custom training wall time: {custom_training_seconds:.2f} s")
print(f"Built-in training wall time: {builtin_training_seconds:.2f} s")

Epoch 1/12
66/66 - 16s - 243ms/step - accuracy: 0.9230 - loss: 0.3145 - val_accuracy: 1.0000 - val_loss: 0.0113 - learning_rate: 0.0010
Epoch 2/12
66/66 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0067 - val_accuracy: 1.0000 - val_loss: 0.0042 - learning_rate: 0.0010
Epoch 3/12
66/66 - 0s - 5ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 0.0023 - learning_rate: 0.0010
Epoch 4/12
66/66 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 0.0014 - learning_rate: 0.0010
Epoch 5/12
66/66 - 0s - 6ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 1.0000 - val_loss: 9.5096e-04 - learning_rate: 0.0010
Epoch 6/12
66/66 - 0s - 6ms/step - accuracy: 1.0000 - loss: 8.0192e-04 - val_accuracy: 1.0000 - val_loss: 6.8369e-04 - learning_rate: 0.0010
Epoch 7/12
66/66 - 0s - 6ms/step - accuracy: 1.0000 - loss: 5.8911e-04 - val_accuracy: 1.0000 - val_loss: 5.1368e-04 - learning_rate: 0.0010
Epoch 8/12
66/66 - 0s - 6ms/step - accuracy: 1

## 14. Evaluate on the untouched test set

Accuracy is included for an intuitive overview. Macro F1, precision, and recall give every class equal weight. The test split is evaluated after training and is not used for checkpoint selection. Latency is measured separately after warm-up to reduce first-call/tracing effects.

In [24]:
def evaluate_classifier(model, model_name):
    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    predicted = probabilities.argmax(axis=1)
    report = classification_report(
        y_test, predicted, labels=np.arange(NUM_CLASSES), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    metrics = {
        "model": model_name,
        "trainable_parameters": int(np.sum([np.prod(v.shape) for v in model.trainable_variables])),
        "test_accuracy": float(accuracy_score(y_test, predicted)),
        "test_macro_f1": float(f1_score(y_test, predicted, average="macro", zero_division=0)),
        "test_macro_precision": float(precision_score(y_test, predicted, average="macro", zero_division=0)),
        "test_macro_recall": float(recall_score(y_test, predicted, average="macro", zero_division=0)),
        "classification_report": report,
        "probabilities": probabilities,
        "predicted": predicted,
    }
    return metrics

custom_metrics = evaluate_classifier(custom_model, "Custom Attention")
builtin_metrics = evaluate_classifier(builtin_model, "Built-In Attention")

metrics_table = pd.DataFrame([
    {k: v for k, v in item.items() if k in {"model", "trainable_parameters", "test_accuracy", "test_macro_f1", "test_macro_precision", "test_macro_recall"}}
    for item in (custom_metrics, builtin_metrics)
])
display(metrics_table)

,model,trainable_parameters,test_accuracy,test_macro_f1,test_macro_precision,test_macro_recall
0,Custom Attention,23189,1.0,1.0,1.0,1.0
1,Built-In Attention,23189,1.0,1.0,1.0,1.0


## 15. Fair inference latency benchmark

The benchmark uses the same held-out batch for both models, warms up each callable, forces device completion by materializing outputs, repeats the measurement, and reports median and interquartile-range time. First-call tracing is outside the measured samples. Results are environment-specific and should not be generalized from one run.

In [25]:
def benchmark_inference(model, batch, repeats=30, warmups=5):
    infer = tf.function(lambda x: model(x, training=False), reduce_retracing=True)
    for _ in range(warmups):
        _ = infer(batch).numpy()
    samples_ms = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = infer(batch)
        _ = result.numpy()  # Materialization synchronizes execution before stopping the timer.
        samples_ms.append((time.perf_counter() - start) * 1000.0)
    samples_ms = np.asarray(samples_ms)
    return {
        "median_batch_ms": float(np.median(samples_ms)),
        "q1_batch_ms": float(np.percentile(samples_ms, 25)),
        "q3_batch_ms": float(np.percentile(samples_ms, 75)),
        "median_per_sample_ms": float(np.median(samples_ms) / int(batch.shape[0])),
        "samples_ms": samples_ms,
    }

benchmark_batch = tf.convert_to_tensor(X_test[:min(BATCH_SIZE, len(X_test))], dtype=tf.int32)
custom_latency = benchmark_inference(custom_model, benchmark_batch)
builtin_latency = benchmark_inference(builtin_model, benchmark_batch)
print("Custom latency:", {k: v for k, v in custom_latency.items() if k != "samples_ms"})
print("Built-in latency:", {k: v for k, v in builtin_latency.items() if k != "samples_ms"})

Custom latency: {'median_batch_ms': 1.6569240001444996, 'q1_batch_ms': 1.5971622499932892, 'q3_batch_ms': 1.7808057501156327, 'median_per_sample_ms': 0.012944718751128903}
Built-in latency: {'median_batch_ms': 1.781635500037737, 'q1_batch_ms': 1.6661954998653528, 'q3_batch_ms': 1.9202480000899413, 'median_per_sample_ms': 0.013919027344044821}


## 16. Training curves and model comparison plots

In [26]:
def plot_history_metric(metric, title, filename):
    fig = go.Figure()
    for label, history in [("Custom Attention", custom_history.history), ("Built-In Attention", builtin_history.history)]:
        if metric in history:
            fig.add_trace(go.Scatter(y=history[metric], mode="lines+markers", name=f"{label} — train"))
        val_key = f"val_{metric}"
        if val_key in history:
            fig.add_trace(go.Scatter(y=history[val_key], mode="lines+markers", line={"dash": "dash"}, name=f"{label} — validation"))
    fig.update_layout(title=title, template=PLOT_TEMPLATE, font_family=PLOT_FONT,
                      xaxis_title="Epoch", yaxis_title=metric.replace("_", " ").title())
    save_plotly_figure(fig, filename)

plot_history_metric("loss", "Training and Validation Loss", "training_loss_curves")
plot_history_metric("accuracy", "Training and Validation Accuracy", "training_accuracy_curves")

In [27]:
long_metrics = metrics_table.melt(id_vars="model", value_vars=["test_accuracy", "test_macro_f1"], var_name="metric", value_name="score")
fig = px.bar(long_metrics, x="metric", y="score", color="model", barmode="group", text_auto=".3f",
             title="Test Accuracy and Macro F1", template=PLOT_TEMPLATE,
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(font_family=PLOT_FONT, yaxis_title="Score", xaxis_title="Metric", yaxis_range=[0, 1.05])
save_plotly_figure(fig, "model_test_metric_comparison")

{'html': 'images/model_test_metric_comparison.html', 'png': None}

In [28]:
pr_rows = []
for item in (custom_metrics, builtin_metrics):
    pr_rows.extend([
        {"model": item["model"], "metric": "Macro precision", "score": item["test_macro_precision"]},
        {"model": item["model"], "metric": "Macro recall", "score": item["test_macro_recall"]},
    ])
fig = px.bar(pd.DataFrame(pr_rows), x="metric", y="score", color="model", barmode="group", text_auto=".3f",
             title="Macro Precision and Recall", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, yaxis_title="Score", yaxis_range=[0, 1.05])
save_plotly_figure(fig, "model_precision_recall_comparison")

{'html': 'images/model_precision_recall_comparison.html', 'png': None}

In [29]:
latency_rows = []
for name, result in [("Custom Attention", custom_latency), ("Built-In Attention", builtin_latency)]:
    latency_rows.append({"model": name, "median_ms": result["median_batch_ms"],
                         "q1_ms": result["q1_batch_ms"], "q3_ms": result["q3_batch_ms"]})
latency_frame = pd.DataFrame(latency_rows)
fig = go.Figure()
for row in latency_frame.itertuples():
    fig.add_trace(go.Bar(name=row.model, x=[row.model], y=[row.median_ms],
                         error_y={"type": "data", "symmetric": False,
                                  "array": [row.q3_ms - row.median_ms],
                                  "arrayminus": [row.median_ms - row.q1_ms]},
                         text=[f"{row.median_ms:.2f} ms"], textposition="outside"))
fig.update_layout(title="Inference Latency (Median, IQR Error Bars)", template=PLOT_TEMPLATE,
                  font_family=PLOT_FONT, yaxis_title="Milliseconds per benchmark batch", showlegend=False)
save_plotly_figure(fig, "model_inference_latency")

{'html': 'images/model_inference_latency.html', 'png': None}

In [30]:
parameter_frame = pd.DataFrame([
    {"model": "Custom Attention", "parameters": custom_metrics["trainable_parameters"]},
    {"model": "Built-In Attention", "parameters": builtin_metrics["trainable_parameters"]},
])
fig = px.bar(parameter_frame, x="model", y="parameters", color="model", text_auto=True,
             title="Trainable Parameter Count", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Model", yaxis_title="Trainable parameters", showlegend=False)
save_plotly_figure(fig, "model_parameter_comparison")

training_time_frame = pd.DataFrame([
    {"model": "Custom Attention", "seconds": custom_training_seconds},
    {"model": "Built-In Attention", "seconds": builtin_training_seconds},
])
fig = px.bar(training_time_frame, x="model", y="seconds", color="model", text_auto=".2f",
             title="Total Training Wall Time", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Model", yaxis_title="Seconds", showlegend=False)
save_plotly_figure(fig, "model_training_time")

{'html': 'images/model_training_time.html', 'png': None}

## 17. Confusion matrices and per-class performance

In [31]:
for item, filename in [(custom_metrics, "confusion_matrix_custom"), (builtin_metrics, "confusion_matrix_builtin")]:
    cm = confusion_matrix(y_test, item["predicted"], labels=np.arange(NUM_CLASSES))
    fig = go.Figure(go.Heatmap(z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues",
                               text=cm, texttemplate="%{text}", hovertemplate="Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>"))
    fig.update_layout(title=f"Confusion Matrix — {item['model']}", template=PLOT_TEMPLATE, font_family=PLOT_FONT,
                      xaxis_title="Predicted class", yaxis_title="Actual class")
    save_plotly_figure(fig, filename)

In [32]:
class_rows = []
for item in (custom_metrics, builtin_metrics):
    report = item["classification_report"]
    for class_name in CLASS_NAMES:
        class_rows.append({"model": item["model"], "class": class_name,
                           "precision": report[class_name]["precision"],
                           "recall": report[class_name]["recall"],
                           "f1-score": report[class_name]["f1-score"]})
per_class = pd.DataFrame(class_rows)
fig = px.bar(per_class, x="class", y="f1-score", color="model", barmode="group", text_auto=".2f",
             title="Per-Class F1 Score", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Class", yaxis_title="F1 score", yaxis_range=[0, 1.05])
save_plotly_figure(fig, "model_per_class_f1")

{'html': 'images/model_per_class_f1.html', 'png': None}

In [33]:
confidence_rows = []
for item in (custom_metrics, builtin_metrics):
    confidence = item["probabilities"].max(axis=1)
    correct = item["predicted"] == y_test
    for value, is_correct in zip(confidence, correct):
        confidence_rows.append({"model": item["model"], "confidence": float(value),
                                "prediction_status": "Correct" if is_correct else "Incorrect"})
confidence_frame = pd.DataFrame(confidence_rows)
fig = px.histogram(confidence_frame, x="confidence", color="prediction_status", facet_col="model",
                   barmode="overlay", opacity=0.7, nbins=25,
                   title="Prediction Confidence: Correct vs Incorrect", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Maximum predicted class probability", yaxis_title="Examples")
save_plotly_figure(fig, "prediction_confidence_distribution")

{'html': 'images/prediction_confidence_distribution.html', 'png': None}

## 18. Visualize attention from trained classifiers

We visualize one held-out test sequence from each model. The padded positions are removed from both axes, and real token strings label the matrix. Attention maps show where the layer distributes probability mass; they should not be read as definitive explanations for why the final classifier predicted a label.

In [34]:
def get_attention_for_model(model, encoded_sequence):
    probs, scores = model(tf.convert_to_tensor(encoded_sequence[None, :], dtype=tf.int32),
                          training=False, return_attention=True)
    scores = scores.numpy()
    if scores.ndim == 4:  # Built-in MHA returns (batch, heads, query, key).
        scores = scores[:, 0, :, :]
    return probs.numpy()[0], scores[0]

example_index = 0
encoded_example = X_test[example_index]
valid_positions = np.where(encoded_example != PAD_ID)[0]
token_labels = [ID_TO_TOKEN.get(int(encoded_example[i]), UNK_TOKEN) for i in valid_positions]
custom_probs_example, custom_attention_example = get_attention_for_model(custom_model, encoded_example)
builtin_probs_example, builtin_attention_example = get_attention_for_model(builtin_model, encoded_example)
custom_valid_attention = custom_attention_example[np.ix_(valid_positions, valid_positions)]
builtin_valid_attention = builtin_attention_example[np.ix_(valid_positions, valid_positions)]

In [35]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Custom attention", "Built-in attention"), shared_yaxes=True)
for col, matrix in [(1, custom_valid_attention), (2, builtin_valid_attention)]:
    fig.add_trace(go.Heatmap(z=matrix, x=token_labels, y=token_labels, colorscale="Turbo",
                             zmin=0, zmax=max(float(custom_valid_attention.max()), float(builtin_valid_attention.max())),
                             hovertemplate="Query token: %{y}<br>Key token: %{x}<br>Probability: %{z:.4f}<extra></extra>",
                             colorbar={"title": "Probability", "x": 1.02} if col == 2 else None,
                             showscale=(col == 2)), row=1, col=col)
fig.update_layout(title=f"Token-to-Token Attention — True label: {ID_TO_CLASS[int(y_test[example_index])]}",
                  template=PLOT_TEMPLATE, font_family=PLOT_FONT, height=550)
fig.update_xaxes(title_text="Key token")
fig.update_yaxes(title_text="Query token")
save_plotly_figure(fig, "attention_heatmap")

{'html': 'images/attention_heatmap.html', 'png': None}

In [36]:
att_diff = np.abs(custom_valid_attention - builtin_valid_attention)
fig = go.Figure(go.Heatmap(z=att_diff, x=token_labels, y=token_labels, colorscale="Magma",
                           hovertemplate="Query: %{y}<br>Key: %{x}<br>Absolute difference: %{z:.6f}<extra></extra>"))
fig.update_layout(title="Custom vs Built-In Attention — Absolute Difference",
                  template=PLOT_TEMPLATE, font_family=PLOT_FONT,
                  xaxis_title="Key token", yaxis_title="Query token")
save_plotly_figure(fig, "attention_comparison")

{'html': 'images/attention_comparison.html', 'png': None}

In [37]:
def normalized_attention_entropy(weights, valid_positions):
    matrix = weights[np.ix_(valid_positions, valid_positions)]
    eps = 1e-12
    row_entropy = -np.sum(matrix * np.log(np.clip(matrix, eps, 1.0)), axis=1)
    denominator = np.log(max(matrix.shape[1], 2))
    return row_entropy / denominator

entropy_rows = []
for model_name, matrix in [("Custom Attention", custom_attention_example), ("Built-In Attention", builtin_attention_example)]:
    values = normalized_attention_entropy(matrix, valid_positions)
    entropy_rows.extend([{"model": model_name, "token": token, "normalized_entropy": float(value)}
                         for token, value in zip(token_labels, values)])
entropy_frame = pd.DataFrame(entropy_rows)
fig = px.bar(entropy_frame, x="token", y="normalized_entropy", color="model", barmode="group",
             title="Normalized Attention Entropy by Query Token", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Query token", yaxis_title="Normalized entropy (0–1)")
save_plotly_figure(fig, "attention_distribution_entropy")

{'html': 'images/attention_distribution_entropy.html', 'png': None}

In [38]:
fig = px.bar(pd.DataFrame({"class": CLASS_NAMES,
                          "custom_probability": custom_probs_example,
                          "builtin_probability": builtin_probs_example}).melt(id_vars="class", var_name="model", value_name="probability"),
             x="class", y="probability", color="model", barmode="group", text_auto=".2f",
             title=f"Prediction Probabilities for Held-Out Example (true: {ID_TO_CLASS[int(y_test[example_index])]})",
             template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Class", yaxis_title="Predicted probability", yaxis_range=[0, 1.05])
save_plotly_figure(fig, "prediction_example_probabilities")

{'html': 'images/prediction_example_probabilities.html', 'png': None}

### Context sensitivity diagnostic

A controlled diagnostic holds a shared cue constant while changing its partner. The displayed probabilities and attention distributions are model outputs for these constructed sequences. They illustrate sensitivity to context, but one example cannot establish a general causal explanation.

In [39]:
def encode_token_list(tokens):
    encoded = np.full((MAX_LEN,), PAD_ID, dtype=np.int32)
    ids = [VOCAB.get(token, UNK_ID) for token in tokens[:MAX_LEN]]
    encoded[:len(ids)] = ids
    return encoded

context_sequences = [
    ["pet", "calm", "dog", "with", "blue", "and", "the", "moon"],
    ["pet", "orbit", "dog", "with", "blue", "and", "the", "moon"],
    ["pet", "harvest", "dog", "with", "blue", "and", "the", "moon"],
]
context_rows = []
context_attention_records = []
for seq_index, tokens in enumerate(context_sequences):
    encoded = encode_token_list(tokens)
    valid_len = len(tokens)
    for model_name, model in [("Custom Attention", custom_model), ("Built-In Attention", builtin_model)]:
        probabilities, weights = get_attention_for_model(model, encoded)
        pred_id = int(np.argmax(probabilities))
        context_rows.append({"sequence": f"Context {seq_index + 1}", "model": model_name,
                             "tokens": " ".join(tokens), "predicted_class": ID_TO_CLASS[pred_id],
                             "confidence": float(np.max(probabilities))})
        attention = weights[:valid_len, :valid_len]
        pet_positions = [i for i, token in enumerate(tokens) if token == "pet"]
        if pet_positions:
            for token, weight in zip(tokens, attention[pet_positions[0]]):
                context_attention_records.append({"sequence": f"Context {seq_index + 1}", "model": model_name,
                                                   "key_token": token, "attention_from_pet": float(weight)})
context_results = pd.DataFrame(context_rows)
display(context_results)
context_attention_df = pd.DataFrame(context_attention_records)
fig = px.bar(context_attention_df, x="key_token", y="attention_from_pet", color="sequence", barmode="group",
             facet_col="model", title="How the Shared Token 'pet' Attends Under Different Contexts",
             template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Key token", yaxis_title="Attention probability")
save_plotly_figure(fig, "attention_context_sensitivity")

,sequence,model,tokens,predicted_class,confidence
0,Context 1,Custom Attention,pet calm dog with blue and the moon,Animals & Pets,0.999798
1,Context 1,Built-In Attention,pet calm dog with blue and the moon,Animals & Pets,0.999716
2,Context 2,Custom Attention,pet orbit dog with blue and the moon,Emotions & Actions,0.999847
3,Context 2,Built-In Attention,pet orbit dog with blue and the moon,Emotions & Actions,0.999771
4,Context 3,Custom Attention,pet harvest dog with blue and the moon,Animals & Pets,0.990729
5,Context 3,Built-In Attention,pet harvest dog with blue and the moon,Food & Fruits,0.982110


{'html': 'images/attention_context_sensitivity.html', 'png': None}

## 19. Comparison report using measured values

The following table is computed from the actual model runs. Training times depend on the hardware and TensorFlow build. Attention output error comes from the dedicated parity test, while classification metrics come from independently trained models and are not expected to match layer-by-layer.

In [40]:
custom_final_loss = float(custom_history.history["loss"][-1])
builtin_final_loss = float(builtin_history.history["loss"][-1])
custom_best_val_loss = float(np.min(custom_history.history["val_loss"]))
builtin_best_val_loss = float(np.min(builtin_history.history["val_loss"]))
comparison_rows = [
    {"Metric": "Attention implementation", "Custom Attention": "Manual TensorFlow ops", "Built-In Attention": "Keras MultiHeadAttention"},
    {"Metric": "Trainable parameters", "Custom Attention": custom_metrics["trainable_parameters"], "Built-In Attention": builtin_metrics["trainable_parameters"]},
    {"Metric": "Final training loss", "Custom Attention": custom_final_loss, "Built-In Attention": builtin_final_loss},
    {"Metric": "Best validation loss", "Custom Attention": custom_best_val_loss, "Built-In Attention": builtin_best_val_loss},
    {"Metric": "Test accuracy", "Custom Attention": custom_metrics["test_accuracy"], "Built-In Attention": builtin_metrics["test_accuracy"]},
    {"Metric": "Test macro F1", "Custom Attention": custom_metrics["test_macro_f1"], "Built-In Attention": builtin_metrics["test_macro_f1"]},
    {"Metric": "Macro precision", "Custom Attention": custom_metrics["test_macro_precision"], "Built-In Attention": builtin_metrics["test_macro_precision"]},
    {"Metric": "Macro recall", "Custom Attention": custom_metrics["test_macro_recall"], "Built-In Attention": builtin_metrics["test_macro_recall"]},
    {"Metric": "Training duration (seconds)", "Custom Attention": custom_training_seconds, "Built-In Attention": builtin_training_seconds},
    {"Metric": "Median inference batch (ms)", "Custom Attention": custom_latency["median_batch_ms"], "Built-In Attention": builtin_latency["median_batch_ms"]},
    {"Metric": "Attention output max abs error (parity test)", "Custom Attention": max_output_error, "Built-In Attention": max_output_error},
    {"Metric": "Implementation complexity", "Custom Attention": "More explicit; easier to inspect", "Built-In Attention": "Higher-level; less maintenance"},
    {"Metric": "Production suitability", "Custom Attention": "Useful for learning / special needs", "Built-In Attention": "Usually preferred for standard use"},
]
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
comparison_table.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

,Metric,Custom Attention,Built-In Attention
0,Attention implementation,Manual TensorFlow ops,Keras MultiHeadAttention
1,Trainable parameters,23189,23189
2,Final training loss,0.000198,0.000328
3,Best validation loss,0.000183,0.000303
4,Test accuracy,1.0,1.0
5,Test macro F1,1.0,1.0
6,Macro precision,1.0,1.0
7,Macro recall,1.0,1.0
8,Training duration (seconds),20.319141,15.802284
9,Median inference batch (ms),1.656924,1.781636


### Interpreting the experiment

- A small accuracy/F1 gap is not automatically meaningful; random initialization, stochastic minibatches, and a finite test set cause variation.
- If parameter counts differ, note it when judging fairness. Even if counts match, low-level implementations may take different kernels.
- The dedicated parity test compares layer outputs under copied weights. Training two independently initialized classifiers is a different experiment and does not imply matching intermediate representations.
- The built-in implementation is usually the pragmatic production default: it is maintained, tested, and can benefit from optimized kernels. A manual layer is excellent for education, debugging, and unusual masking or scoring logic.
- Never generalize runtime findings beyond the reported device, TensorFlow version, sequence length, batch size, and measurement procedure.

## 20. Configurable ablation studies

These experiments are intentionally optional/lightweight. They benchmark forward-pass attention operations rather than retraining six separate classifiers. Change the settings below to explore larger dimensions or sequence lengths. Attention score memory scales as \(O(T^2)\) per head, which is why long sequences can become expensive.

In [41]:
RUN_ABLATIONS = True
ABLATION_BATCH = 16
ABLATION_LENGTHS = [16, 32, 64, 128]
ABLATION_DIMS = [32, 64, 128]
ABLATION_REPEATS = 8


def time_attention_layer(layer, x, valid_mask, repeats=ABLATION_REPEATS):
    # A short warm-up is excluded from the measured samples.
    for _ in range(2):
        result = layer(x, valid_tokens=valid_mask)
        _ = result.numpy() if tf.is_tensor(result) else result[0].numpy()
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = layer(x, valid_tokens=valid_mask)
        if tf.is_tensor(result):
            _ = result.numpy()
        else:
            _ = result[0].numpy()
        samples.append((time.perf_counter() - start) * 1000)
    return float(np.median(samples))

length_benchmark_rows = []
if RUN_ABLATIONS:
    for length in ABLATION_LENGTHS:
        x = tf.random.normal((ABLATION_BATCH, length, 32), seed=SEED + length)
        mask = tf.ones((ABLATION_BATCH, length), dtype=tf.bool)
        manual = SelfAttention(32, key_dim=32, value_dim=32)
        builtin = BuiltinSelfAttention(32, key_dim=32, value_dim=32)
        manual_ms = time_attention_layer(manual, x, mask)
        builtin_ms = time_attention_layer(builtin, x, mask)
        length_benchmark_rows.extend([
            {"sequence_length": length, "implementation": "Manual", "median_ms": manual_ms},
            {"sequence_length": length, "implementation": "Built-in", "median_ms": builtin_ms},
        ])
    length_benchmark = pd.DataFrame(length_benchmark_rows)
    display(length_benchmark)
    fig = px.line(length_benchmark, x="sequence_length", y="median_ms", color="implementation", markers=True,
                  title="Attention Runtime vs Sequence Length (Forward Pass)", template=PLOT_TEMPLATE)
    fig.update_layout(font_family=PLOT_FONT, xaxis_title="Sequence length", yaxis_title="Median milliseconds")
    save_plotly_figure(fig, "ablation_sequence_length")

,sequence_length,implementation,median_ms
0,16,Manual,11.191531
1,16,Built-in,10.895764
2,32,Manual,9.975338
3,32,Built-in,11.679194
4,64,Manual,11.418388
5,64,Built-in,11.954329
6,128,Manual,11.429034
7,128,Built-in,11.609688


In [42]:
dimension_rows = []
if RUN_ABLATIONS:
    for dim in ABLATION_DIMS:
        length = 32
        x = tf.random.normal((ABLATION_BATCH, length, dim), seed=SEED + dim)
        mask = tf.ones((ABLATION_BATCH, length), dtype=tf.bool)
        manual = SelfAttention(dim, key_dim=dim, value_dim=dim)
        builtin = BuiltinSelfAttention(dim, key_dim=dim, value_dim=dim)
        dimension_rows.extend([
            {"embedding_dimension": dim, "implementation": "Manual", "median_ms": time_attention_layer(manual, x, mask)},
            {"embedding_dimension": dim, "implementation": "Built-in", "median_ms": time_attention_layer(builtin, x, mask)},
        ])
    dimension_benchmark = pd.DataFrame(dimension_rows)
    fig = px.bar(dimension_benchmark, x="embedding_dimension", y="median_ms", color="implementation", barmode="group",
                 title="Attention Runtime vs Embedding Dimension", template=PLOT_TEMPLATE)
    fig.update_layout(font_family=PLOT_FONT, xaxis_title="Embedding dimension", yaxis_title="Median milliseconds")
    save_plotly_figure(fig, "ablation_embedding_dimension")

In [43]:
# Scaling diagnostic: compare softmax entropy for scaled and unscaled random logits.
rng = np.random.default_rng(SEED)
scale_dims = [8, 32, 64, 128]
scale_rows = []
for dim in scale_dims:
    logits = rng.normal(size=(512, 12, 12)).astype(np.float32) * np.sqrt(dim)
    for label, values in [("Scaled", logits / np.sqrt(dim)), ("Unscaled", logits)]:
        shifted = values - values.max(axis=-1, keepdims=True)
        p = np.exp(shifted) / np.exp(shifted).sum(axis=-1, keepdims=True)
        entropy = -np.sum(p * np.log(np.clip(p, 1e-12, 1.0)), axis=-1).mean()
        scale_rows.append({"key_dimension": dim, "condition": label, "mean_entropy": float(entropy)})
scale_frame = pd.DataFrame(scale_rows)
fig = px.line(scale_frame, x="key_dimension", y="mean_entropy", color="condition", markers=True,
              title="Scaling and Softmax Entropy on Synthetic Logits", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Key dimension", yaxis_title="Mean entropy (nats)")
save_plotly_figure(fig, "ablation_attention_scaling")

{'html': 'images/ablation_attention_scaling.html', 'png': None}

In [44]:
# Padding-mask diagnostic: a pad key should receive zero attention when masked, but not necessarily when unmasked.
mask_demo_q = tf.random.normal((1, 4, 8), seed=SEED + 300)
mask_demo_k = tf.random.normal((1, 4, 8), seed=SEED + 301)
mask_demo_v = tf.random.normal((1, 4, 8), seed=SEED + 302)
mask_demo = tf.constant([[True, True, False, False]])
_, weights_masked = attention_math(mask_demo_q, mask_demo_k, mask_demo_v, key_mask=mask_demo)
_, weights_unmasked = attention_math(mask_demo_q, mask_demo_k, mask_demo_v, key_mask=None)
mask_compare = pd.DataFrame({
    "key_position": ["Token 1", "Token 2", "PAD 1", "PAD 2"],
    "With padding mask": weights_masked.numpy()[0, 0],
    "Without mask (diagnostic only)": weights_unmasked.numpy()[0, 0],
})
fig = px.bar(mask_compare, x="key_position", y=["With padding mask", "Without mask (diagnostic only)"], barmode="group",
             title="Masking Prevents Attention to Padding Keys", template=PLOT_TEMPLATE)
fig.update_layout(font_family=PLOT_FONT, xaxis_title="Key position", yaxis_title="Attention probability")
save_plotly_figure(fig, "ablation_padding_mask")

{'html': 'images/ablation_padding_mask.html', 'png': None}

In [45]:
# Optional built-in multi-head extension. This is illustrative and is not part of the single-head fairness comparison.
MULTIHEAD_COUNT = 4
if EMBED_DIM % MULTIHEAD_COUNT == 0:
    multihead_layer = tf.keras.layers.MultiHeadAttention(
        num_heads=MULTIHEAD_COUNT, key_dim=EMBED_DIM // MULTIHEAD_COUNT,
        dropout=0.0, name="optional_four_head_attention"
    )
    mh_x = tf.random.normal((1, 8, EMBED_DIM), seed=SEED + 401)
    mh_mask = tf.ones((1, 8), dtype=tf.bool)
    mh_attn_mask = tf.broadcast_to(mh_mask[:, None, :], (1, 8, 8))
    mh_output, mh_scores = multihead_layer(mh_x, mh_x, attention_mask=mh_attn_mask,
                                           return_attention_scores=True, training=False)
    print("Multi-head output shape:", mh_output.shape)
    print("Multi-head score shape:", mh_scores.shape)
    fig = make_subplots(rows=2, cols=2, subplot_titles=[f"Head {i+1}" for i in range(MULTIHEAD_COUNT)])
    for head in range(MULTIHEAD_COUNT):
        row, col = divmod(head, 2)
        fig.add_trace(go.Heatmap(z=mh_scores.numpy()[0, head], colorscale="Turbo", showscale=False), row=row+1, col=col+1)
    fig.update_layout(title="Optional Multi-Head Attention — Four Different Projection Subspaces",
                      template=PLOT_TEMPLATE, font_family=PLOT_FONT, height=650)
    save_plotly_figure(fig, "optional_multihead_attention")

Multi-head output shape: (1, 8, 48)
Multi-head score shape: (1, 4, 8, 8)


## 21. Export predictions, metrics, histories, summaries, and models

The export cell saves measured results only. Predictions include each test example's true class, predicted class, confidence, and per-class probability. Metrics include runtime environment, split sizes, parameter counts, training time, evaluation metrics, and benchmark timings. The two models are saved in Keras' `.keras` format.

In [46]:
prediction_rows = []
for i, source_index in enumerate(test_df.index):
    row = {
        "sample_id": int(raw_df.loc[source_index, "sample_id"]),
        "true_label": ID_TO_CLASS[int(y_test[i])],
        "custom_predicted_label": ID_TO_CLASS[int(custom_metrics["predicted"][i])],
        "custom_confidence": float(custom_metrics["probabilities"][i].max()),
        "builtin_predicted_label": ID_TO_CLASS[int(builtin_metrics["predicted"][i])],
        "builtin_confidence": float(builtin_metrics["probabilities"][i].max()),
    }
    for class_id, class_name in enumerate(CLASS_NAMES):
        row[f"custom_prob_{class_id}"] = float(custom_metrics["probabilities"][i, class_id])
        row[f"builtin_prob_{class_id}"] = float(builtin_metrics["probabilities"][i, class_id])
    prediction_rows.append(row)
predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)

metrics_export = {
    "status": "completed",
    "dataset": {
        "kind": "synthetic context-pair sequence classification",
        "n_samples": int(len(raw_df)),
        "class_names": CLASS_NAMES,
        "vocabulary_size_including_special_tokens": int(len(VOCAB)),
        "max_sequence_length": int(MAX_LEN),
        "split_sizes": {"train": int(len(train_df)), "validation": int(len(val_df)), "test": int(len(test_df))},
        "class_counts": {str(k): int(v) for k, v in raw_df["label"].value_counts().items()},
    },
    "environment": {
        "python": platform.python_version(), "platform": platform.platform(),
        "tensorflow": tf.__version__, "numpy": np.__version__, "pandas": pd.__version__,
        "plotly": plotly.__version__, "devices": [d.name for d in tf.config.list_physical_devices()],
    },
    "training_config": {"seed": SEED, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
                        "learning_rate": LEARNING_RATE, "embedding_dim": EMBED_DIM, "feed_forward_dim": FF_DIM},
    "custom_attention": {
        "trainable_parameters": custom_metrics["trainable_parameters"],
        "final_train_loss": custom_final_loss, "best_validation_loss": custom_best_val_loss,
        "training_seconds": float(custom_training_seconds), "test_accuracy": custom_metrics["test_accuracy"],
        "test_macro_f1": custom_metrics["test_macro_f1"],
        "test_macro_precision": custom_metrics["test_macro_precision"], "test_macro_recall": custom_metrics["test_macro_recall"],
        "inference_latency": {k: v for k, v in custom_latency.items() if k != "samples_ms"},
        "classification_report": custom_metrics["classification_report"],
    },
    "builtin_attention": {
        "trainable_parameters": builtin_metrics["trainable_parameters"],
        "final_train_loss": builtin_final_loss, "best_validation_loss": builtin_best_val_loss,
        "training_seconds": float(builtin_training_seconds), "test_accuracy": builtin_metrics["test_accuracy"],
        "test_macro_f1": builtin_metrics["test_macro_f1"],
        "test_macro_precision": builtin_metrics["test_macro_precision"], "test_macro_recall": builtin_metrics["test_macro_recall"],
        "inference_latency": {k: v for k, v in builtin_latency.items() if k != "samples_ms"},
        "classification_report": builtin_metrics["classification_report"],
    },
    "parity_test": {"status": parity_status, "tolerance": 2e-5,
                    "max_output_abs_error": max_output_error, "mean_output_abs_error": mean_output_error,
                    "max_attention_abs_error": max_attention_error, "mean_attention_abs_error": mean_attention_error},
    "caveats": [
        "Synthetic results do not establish real-world NLP performance.",
        "Latency results depend on hardware, runtime, model shape, batch size, and implementation details.",
        "Attention weights should not be treated as definitive explanations of model reasoning."
    ],
}
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_export, file, indent=2, allow_nan=False)

custom_model.save(OUTPUT_DIR / "custom_attention_model.keras")
builtin_model.save(OUTPUT_DIR / "builtin_attention_model.keras")

summary_content = "\n\n".join(summary_lines)
summary_content += f"\n\nEnvironment: TensorFlow {tf.__version__}; Python {platform.python_version()}"
summary_content += f"\nParity test: {parity_status}"
summary_content += f"\nCustom training seconds: {custom_training_seconds:.4f}"
summary_content += f"\nBuilt-in training seconds: {builtin_training_seconds:.4f}"
(OUTPUT_DIR / "model_summary.txt").write_text(summary_content, encoding="utf-8")

# Save generated examples for reproducible inspection; no external files are required.
raw_export = raw_df.copy()
raw_export["tokens"] = raw_export["tokens"].map(lambda tokens: " ".join(tokens))
raw_export.to_csv(DATA_DIR / "synthetic_sequences.csv", index=False)
print("Exported artifacts:")
for path in sorted(list(OUTPUT_DIR.iterdir()) + list(IMAGE_DIR.iterdir()) + list(DATA_DIR.iterdir())):
    print(" -", path.relative_to(PROJECT_DIR))

Exported artifacts:
 - data/synthetic_sequences.csv
 - images/ablation_attention_scaling.html
 - images/ablation_embedding_dimension.html
 - images/ablation_padding_mask.html
 - images/ablation_sequence_length.html
 - images/attention_comparison.html
 - images/attention_context_sensitivity.html
 - images/attention_distribution_entropy.html
 - images/attention_heatmap.html
 - images/attention_numerical_walkthrough.html
 - images/attention_output_agreement.html
 - images/attention_output_error_distribution.html
 - images/attention_weight_difference.html
 - images/confusion_matrix_builtin.html
 - images/confusion_matrix_custom.html
 - images/dataset_class_distribution.html
 - images/dataset_example_sequences.html
 - images/dataset_sequence_lengths.html
 - images/dataset_split_sizes.html
 - images/dataset_token_frequency.html
 - images/model_inference_latency.html
 - images/model_parameter_comparison.html
 - images/model_per_class_f1.html
 - images/model_precision_recall_comparison.html
 -

## 22. Automated validation checklist

These checks are assertions on real runtime objects and saved outputs. A failed assertion should be investigated rather than bypassed. PNG images are optional; an HTML file is the required Plotly artifact.

In [47]:
validation_results = []
def record_check(name, condition, detail=""):
    passed = bool(condition)
    validation_results.append({"check": name, "passed": passed, "detail": detail})
    print(("PASS" if passed else "FAIL") + f" — {name}" + (f": {detail}" if detail else ""))
    if not passed:
        raise AssertionError(f"Validation failed: {name}. {detail}")

record_check("dataset row count", len(raw_df) == 12_000, f"observed={len(raw_df)}")
record_check("valid class labels", set(np.unique(y_train)) == set(range(NUM_CLASSES)))
record_check("padding token is zero", VOCAB[PAD_TOKEN] == 0)
record_check("encoded tensor shape", X_train.shape[1] == MAX_LEN)
record_check("attention probabilities sum to one on valid keys",
             np.allclose(tf.reduce_sum(weights_test, axis=-1).numpy(), 1.0, atol=1e-6))
record_check("masked key probability negligible", float(tf.reduce_max(masked_probs)) < 1e-7)
record_check("manual reference parity", manual_error < 1e-6, f"max_error={manual_error:.8g}")
record_check("metric values finite", np.isfinite([custom_metrics["test_accuracy"], builtin_metrics["test_accuracy"]]).all())
record_check("HTML visualization exists", any(IMAGE_DIR.glob("*.html")), f"html_count={len(list(IMAGE_DIR.glob('*.html')))}")
record_check("predictions CSV exists", (OUTPUT_DIR / "predictions.csv").exists())
record_check("metrics JSON exists", (OUTPUT_DIR / "metrics.json").exists())
record_check("model summary exists", (OUTPUT_DIR / "model_summary.txt").exists())
display(pd.DataFrame(validation_results))

PASS — dataset row count: observed=12000
PASS — valid class labels
PASS — padding token is zero
PASS — encoded tensor shape
PASS — attention probabilities sum to one on valid keys
PASS — masked key probability negligible
PASS — manual reference parity: max_error=0
PASS — metric values finite
PASS — HTML visualization exists: html_count=30
PASS — predictions CSV exists
PASS — metrics JSON exists
PASS — model summary exists


,check,passed,detail
0,dataset row count,True,observed=12000
1,valid class labels,True,
2,padding token is zero,True,
3,encoded tensor shape,True,
4,attention probabilities sum to one on valid keys,True,
5,masked key probability negligible,True,
6,manual reference parity,True,max_error=0
7,metric values finite,True,
8,HTML visualization exists,True,html_count=30
9,predictions CSV exists,True,


## 23. Final conclusions and further learning

### What to take away

1. Self-attention turns projected token representations into context-sensitive representations through similarity scores, softmax probabilities, and weighted value aggregation.
2. Scaling and padding masks are not implementation details to hand-wave away; they materially affect numerical behavior.
3. A manual implementation makes every operation inspectable. Keras' built-in layer is typically the better default for maintainable production systems unless custom behavior is needed.
4. Numerical parity and predictive performance answer different questions. Parity asks whether aligned layer calculations agree; training asks whether independent models learn the task.
5. Synthetic experiments are controlled teaching tools. Follow up with real language benchmarks, careful splits, and task-specific error analysis before making claims about practical NLP performance.

### Further learning

- Vaswani et al., *Attention Is All You Need* (2017): https://arxiv.org/abs/1706.03762
- TensorFlow `tf.keras.layers.MultiHeadAttention`: https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention
- TensorFlow tutorial on Transformer models: https://www.tensorflow.org/text/tutorials/transformer
- Jay Alammar, *The Illustrated Transformer*: https://jalammar.github.io/illustrated-transformer/

### Reproducibility notes

Set the random seeds to rerun the demonstration, but note that bitwise identity is not guaranteed across devices, TensorFlow/CUDA versions, or nondeterministic kernels. Keep the generated metrics JSON with the environment fields when sharing performance results.